In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

In [2]:
df = pd.read_csv("sample_trading_data.csv")

In [3]:
df_aug = df.sample(n=len(df), replace=True).copy()

In [4]:
raw_numeric_cols = [
    'lot_size',
    'entry_price',
    'exit_price',
    'stop_loss',
    'take_profit',
    'pnl'
]

for col in raw_numeric_cols:
    noise = np.random.normal(0, 0.01, size=len(df_aug))  # smaller noise for raw data
    df_aug[col] = df_aug[col] + noise

In [5]:
df_combined = pd.concat([df, df_aug], ignore_index=True)

print("Dataset size after augmentation:", len(df_combined))

Dataset size after augmentation: 200


In [6]:
df_combined['open_time'] = pd.to_datetime(df_combined['open_time'])
df_combined['close_time'] = pd.to_datetime(df_combined['close_time'])

In [7]:
df_combined['trade_duration'] = (
    df_combined['close_time'] - df_combined['open_time']
).dt.total_seconds()

df_combined['open_hour'] = df_combined['open_time'].dt.hour
df_combined['day_of_week'] = df_combined['open_time'].dt.dayofweek

In [8]:
df_combined['price_change'] = (
    df_combined['exit_price'] - df_combined['entry_price']
)

df_combined['return_pct'] = (
    (df_combined['exit_price'] - df_combined['entry_price']) /
    df_combined['entry_price']
)

df_combined['risk'] = abs(
    df_combined['entry_price'] - df_combined['stop_loss']
)

df_combined['reward'] = abs(
    df_combined['take_profit'] - df_combined['entry_price']
)

df_combined['rr_ratio'] = df_combined['reward'] / (df_combined['risk'] + 1e-6)

df_combined['profit_per_lot'] = (
    df_combined['pnl'] / (df_combined['lot_size'] + 1e-6)
)

df_combined['is_profit'] = (df_combined['pnl'] > 0).astype(int)

df_combined['direction_encoded'] = df_combined['direction'].map({
    'BUY': 1,
    'SELL': 0
})

In [9]:

numeric_cols = [
    'lot_size','entry_price','exit_price',
    'stop_loss','take_profit','pnl',
    'trade_duration','open_hour','day_of_week',
    'price_change','return_pct','risk','reward',
    'rr_ratio','profit_per_lot'
]

scaler = RobustScaler()
df_combined[numeric_cols] = scaler.fit_transform(df_combined[numeric_cols])

In [13]:
df_combined.head()

,user_id,trade_id,open_time,close_time,pair,direction,lot_size,entry_price,exit_price,stop_loss,...,open_hour,day_of_week,price_change,return_pct,risk,reward,rr_ratio,profit_per_lot,is_profit,direction_encoded
0,TRADER001,1,2026-01-15 09:30:00,2026-01-15 10:15:00,EUR/USD,BUY,0.00000,-0.001220,-0.001169,-0.001282,...,-0.2,-0.5,0.054182,0.226319,-0.019911,-0.011847,0.024443,0.024143,1,1
1,TRADER001,2,2026-01-15 10:45:00,2026-01-15 11:02:00,EUR/USD,SELL,1.21783,-0.001186,-0.001223,-0.001181,...,0.0,-0.5,-0.498469,-1.171724,-0.019911,-0.001892,1.024048,-1.022558,0,0
2,TRADER001,3,2026-01-15 11:05:00,2026-01-15 11:28:00,GBP/JPY,BUY,4.87132,1.242078,1.232241,1.247362,...,0.2,-0.5,-55.423446,-1.132950,0.980089,4.945417,3.024246,-0.921266,0,1
3,TRADER001,4,2026-01-15 14:20:00,2026-01-15 15:10:00,EUR/USD,BUY,0.00000,-0.001216,-0.001162,-0.001279,...,0.8,-0.5,0.075438,0.279969,-0.019911,-0.011847,0.024443,0.074789,1,1
4,TRADER001,5,2026-01-15 15:45:00,2026-01-15 16:30:00,USD/JPY,SELL,0.00000,0.997349,1.000527,1.002251,...,1.0,-0.5,25.348579,0.157650,0.980089,0.963679,-0.974963,-1.090085,0,0


In [17]:
print(df_combined.columns)

print("\nFinal Dataset Size:", len(df_combined))

Index(['user_id', 'trade_id', 'open_time', 'close_time', 'pair', 'direction',
       'lot_size', 'entry_price', 'exit_price', 'stop_loss', 'take_profit',
       'pnl', 'trade_duration', 'open_hour', 'day_of_week', 'price_change',
       'return_pct', 'risk', 'reward', 'rr_ratio', 'profit_per_lot',
       'is_profit', 'direction_encoded'],
      dtype='object')

Final Dataset Size: 200


In [18]:
df_combined.to_csv("final_pipeline_dataset.csv", index=False)